# Homework 4: Airflow

## Task 2: Generate BERT embeddings

In [1]:
import os
import re
import zipfile
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import boto3
from transformers import AutoTokenizer, AutoModel

/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
BUCKET_NAME = "de300-airflow-kwon-ha-gong"
DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

LOCAL_ZIP = "ml-1m.zip"
EXTRACT_TO = "data/"
MOVIES_DAT_PATH = "data/ml-1m/movies.dat"

LOCAL_PT_OUTPUT = "movie_embeddings_full.pt"
S3_ZIP_KEY = "data/ml-1m.zip"
S3_PT_KEY = "data/movie_embeddings_full.pt"

MODEL_NAME = "distilbert-base-uncased"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

s3_client = boto3.client('s3')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def check_bucket_for_dataset(bucket_name, file_key):
    try: 
        s3_client.head_object(Bucket=bucket_name, Key=file_key)
        return True
    except: 
        return False
        
def download_dataset(bucket_name, dataset_name, dataset_url, file_key):
    print("Downloading dataset.")
    
    response = requests.get(dataset_url)
    with open(dataset_name, 'wb') as f:
        f.write(response.content)
        
    print("Uploading raw zip file to S3 bucket.")
    s3_client.upload_file(dataset_name, bucket_name, file_key)

def extract_dataset_locally(local_zip, extract_to):
    print("Extracting zip.")
    
    with zipfile.ZipFile(local_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        
    print("Data extraction process is done.")

def get_year_and_fix_title(row):
    original_title = row['title']
    
    is_year = re.search(r'\((\d{4})\)', original_title)
    if is_year:
        year = int(is_year.group(1)) 
    else:
        year = None
    
    clean_title = re.sub(r'\s\(\d{4}\)$', '', original_title)
    
    return pd.Series([clean_title, year])

def load_all_movies(file_path):
    print("Loading and processing all movies from the data file")
    movies_df = pd.read_csv(
        file_path, 
        sep='::',  
        engine='python', 
        encoding='latin-1',
        names=['movie_id', 'title', 'genres']
    )
    movies_df[['clean_title', 'year']] = movies_df.apply(get_year_and_fix_title, axis=1)
    return movies_df

def build_movie_text(row):
    genres_string = str(row['genres'])
    clean_genres = genres_string.replace('|', ' ')
    
    title = row['clean_title']
    year = row['year']
    
    if pd.notnull(year):
        year_str = f"({int(year)})"
    else:
        year_str = ""
        
    return f"{title}. {year_str}. {clean_genres}"

@torch.no_grad()
def bert_embed(texts, max_len=256, batch_size=64):
    encoder.eval()
    
    all_embeddings = []
    total_texts = len(texts)
    print("Generating BERT embeddings for all movies in batches.")
    
    # have to generate in chunks or else RAM will run out (ran into issue) 
    for i in range(0, total_texts, batch_size):
        batch_texts = texts[i:i + batch_size]
        
        batch = tokenizer(
            batch_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
        )
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        
        out = encoder(**batch)
        cls = out.last_hidden_state[:, 0]
        emb = F.normalize(cls, dim=-1)
        
        # append the batches together 
        all_embeddings.append(emb.cpu().numpy().astype("float32"))
  
    return np.vstack(all_embeddings)
    
def embeddings_main():
    # check if embeddings are already in S3
    if check_bucket_for_dataset(BUCKET_NAME, S3_PT_KEY):
        print("BERT embeddings have already been generated and are in S3.")
        return  

    # Get the raw file 
    if not check_bucket_for_dataset(BUCKET_NAME, S3_ZIP_KEY):
        download_dataset(BUCKET_NAME, LOCAL_ZIP, DATASET_URL, S3_ZIP_KEY)
        print("Downloaded dataset.")
    else: 
        print("Dataset already exists in S3 so gettin ga copy.")
        if not os.path.exists(LOCAL_ZIP):
            s3_client.download_file(BUCKET_NAME, S3_ZIP_KEY, LOCAL_ZIP)

    extract_dataset_locally(LOCAL_ZIP, EXTRACT_TO)

    # process all movies
    movies_df_full = load_all_movies(MOVIES_DAT_PATH)
    movies_df_full['bert_text'] = movies_df_full.apply(build_movie_text, axis=1)
    
    item_vecs = bert_embed(movies_df_full['bert_text'].tolist(), batch_size=64)

    output_data_full = {
        'embeddings': item_vecs,
        'movie_ids': movies_df_full['movie_id'].values,
        'titles': movies_df_full['clean_title'].values,
        'years': movies_df_full['year'].values
    }

    # save everything to S3
    torch.save(output_data_full, LOCAL_PT_OUTPUT)
    try:
        s3_client.upload_file(LOCAL_PT_OUTPUT, BUCKET_NAME, S3_PT_KEY)
        print("Successfully processed and uploaded embeddings to S3.")
    except Exception as e:
        print("Error while uploading embeddings to S3")

In [4]:
embeddings_main()

BERT embeddings have already been generated and are in S3.


## Task 3: Split the data

In [5]:
RATINGS_DAT_PATH = "data/ml-1m/ratings.dat"

# Get the data from S3 and put it into dataframe
def load_ratings_data(file_path):
    print("Loading ratings data.")
    columns = ['user_id', 'movie_id', 'rating', 'timestamp']
    ratings_df = pd.read_csv(
        file_path, 
        sep='::', 
        engine='python', 
        names=columns,
        encoding='latin-1'
    )
    
    ratings_df['datetime'] = pd.to_datetime(ratings_df['timestamp'], unit='s')
    return ratings_df

# Split the df into 4 chunks based on specified dates
def split_into_partitions(ratings_df):
    print("Partitioning data.")

    part_1_start = "2000-04-25 00:00:00"
    part_1_end = "2000-08-03 23:59:59"
    part_2_end = "2000-10-31 23:59:59"
    part_3_end = "2000-11-26 23:59:59"
    
    # partition 1
    df_part1 = ratings_df[
        (ratings_df['datetime'] >= part_1_start) & 
        (ratings_df['datetime'] <= part_1_end)
    ]
    
    # partition 2
    df_part2 = ratings_df[
        (ratings_df['datetime'] > part_1_end) & 
        (ratings_df['datetime'] <= part_2_end)
    ]
    
    # partition 3
    df_part3 = ratings_df[
        (ratings_df['datetime'] > part_2_end) & 
        (ratings_df['datetime'] <= part_3_end)
    ]
    
    # partition 4
    df_part4 = ratings_df[ratings_df['datetime'] > part_3_end]
    
    return [df_part1, df_part2, df_part3, df_part4]

def save_and_upload_partitions(partitions_list, bucket_name):
    for index, df_chunk in enumerate(partitions_list):
        part_number = index + 1
        local_filename = f"ratings_part{part_number}.csv"
        s3_key = f"data/partitions/ratings_part{part_number}.csv"
                
        final_csv_df = df_chunk.drop(columns=['datetime'])
        final_csv_df.to_csv(local_filename, index=False)
        
        s3_client.upload_file(local_filename, bucket_name, s3_key)
        print("Uploaded partition to S3")

def partitions_main():
    full_ratings = load_ratings_data(RATINGS_DAT_PATH)
    data_partitions = split_into_partitions(full_ratings)
    save_and_upload_partitions(data_partitions, BUCKET_NAME)
    
    print("All 4 partitions are generated and in S3. All processes done.")

In [6]:
partitions_main()

Loading ratings data.
Partitioning data.
Uploaded partition to S3
Uploaded partition to S3
Uploaded partition to S3
Uploaded partition to S3
All 4 partitions are generated and in S3. All processes done.
